# 🖼️🎬 Kaggle Image & Video AI Studio
### Dịch vụ Sinh ảnh & Video AI chuyên biệt:
- 🖼️ **Image**: Qwen-Image-2.1 GGUF qua ComfyUI Backend
- 🎬 **Video**: Wan2.2 TI2V / T2V Diffusion qua ComfyUI Backend
- 🌐 **Chuẩn REST API**: OpenAI , 


In [ ]:
# 1. Nạp biến môi trường tự động (Hỗ trợ Kaggle Secrets, file .env, hoặc cấu hình runtime)
import os, json, urllib.request

def _ntfy_ping(msg: str):
    for t in ['studio-ai-url-cuongnguyen1802', 'studio-ai-debug-cuongnguyen1802']:
        try:
            req = urllib.request.Request(f'https://ntfy.sh/{t}', data=msg.encode('utf-8'), headers={'Title': 'Visual Studio Boot'}, method='POST')
            urllib.request.urlopen(req, timeout=4)
        except Exception:
            pass

_ntfy_ping('🚀 [BOOT] Cell 1 execution started!')


# Thiết lập tối ưu GPU & Cache
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["IDLE_SHUTDOWN_MINUTES"] = "120"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["STUDIO_MODE"] = "image_video"
os.environ["HF_HOME"] = "/tmp/hf_cache"

# Nạp Kaggle UserSecrets nếu được cấp quyền
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for key in ["KAGGLE_USERNAME", "KAGGLE_KEY", "HF_TOKEN", "NTFY_TOPIC"]:
        val = secrets.get_secret(key)
        if val:
            os.environ[key] = val
except Exception:
    pass

# Đọc file .env nếu có sẵn
for env_candidate in [".env", "/kaggle/working/Gen_Image-Video/kaggle/all-in-one/.env"]:
    if os.path.exists(env_candidate):
        try:
            with open(env_candidate, "r") as ef:
                for line in ef:
                    line = line.strip()
                    if line and not line.startswith("#") and "=" in line:
                        k, v = line.split("=", 1)
                        clean_val = v.strip().strip("\"").strip("\x27")
                        os.environ.setdefault(k.strip(), clean_val)
        except Exception:
            pass

# Tạo cấu hình ~/.kaggle/kaggle.json để Kaggle API / CLI hoạt động thông suốt
try:
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as kf:
            json.dump({
                "username": os.environ["KAGGLE_USERNAME"],
                "key": os.environ["KAGGLE_KEY"]
            }, kf)
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
except Exception:
    pass

print(f"✅ Môi trường cấu hình hoàn tất! User: {os.environ.get('KAGGLE_USERNAME', 'Chưa cấu hình')}")


In [ ]:
# 2. Đồng bộ mã nguồn Studio AI (Live Payload)
import os, base64, io, tarfile

target_dir = "/kaggle/working/Gen_Image-Video/kaggle/all-in-one"
os.makedirs(target_dir, exist_ok=True)
os.chdir(target_dir)

payload_data = "..."
raw_tar = base64.b64decode(payload_data)
with tarfile.open(fileobj=io.BytesIO(raw_tar), mode="r:gz") as tar:
    tar.extractall(".")

print("✅ Toàn bộ module Studio AI phiên bản 3.5 đã đồng bộ thành công!")

In [ ]:
# --- CELL 3: SETUP ENVIRONMENT (100% Offline via kaggle-image-video-wheels + Safe Fallback) ---
import glob, os, shutil, subprocess, sys, time, urllib.request, torch

def _ntfy_dbg(msg: str):
    try:
        for topic in ["studio-ai-url-cuongnguyen1802", "studio-ai-url-nguynxuncngde180528", "studio-ai-debug-cuongnguyen1802"]:
            req = urllib.request.Request(
                f"https://ntfy.sh/{topic}",
                data=msg.encode("utf-8"),
                headers={"Title": "Visual Studio Setup", "Priority": "3"},
                method="POST"
            )
            try:
                urllib.request.urlopen(req, timeout=4)
            except Exception:
                pass
    except Exception:
        pass

t_setup_0 = time.time()
_ntfy_dbg("[IMG_VID] Cell 3 starting environment setup...")

# Tìm bundle trực tiếp không cần recursive scan
bundle_candidates = [
    "/kaggle/input/kaggle-image-video-wheels/image_video_runtime_bundle.tar.bin",
    "/kaggle/input/cuongnguyen1802/kaggle-image-video-wheels/image_video_runtime_bundle.tar.bin",
]
bundle_path = None
for cp in bundle_candidates:
    if os.path.exists(cp):
        bundle_path = cp
        break

if not bundle_path:
    for d in glob.glob("/kaggle/input/*"):
        cand = os.path.join(d, "image_video_runtime_bundle.tar.bin")
        if os.path.exists(cand):
            bundle_path = cand
            break

if bundle_path:
    print(f"🚀 [Offline Mode] Đang giải nén ComfyUI + GGUF + VHS & Wheels từ: {bundle_path}")
    _ntfy_dbg(f"[IMG_VID] Unpacking offline bundle from {bundle_path}...")
    rt_dir = "/tmp/imgvid_rt"
    shutil.rmtree(rt_dir, ignore_errors=True)
    os.makedirs(rt_dir, exist_ok=True)
    subprocess.run(["tar", "-xzf", bundle_path, "-C", rt_dir], check=True)

    if os.path.exists("/kaggle/working/ComfyUI"):
        shutil.rmtree("/kaggle/working/ComfyUI", ignore_errors=True)
    shutil.move(f"{rt_dir}/ComfyUI", "/kaggle/working/ComfyUI")

    print("🔄 Cập nhật ComfyUI & GGUF nodes mới nhất (hỗ trợ MiniMax H3)...")
    _ntfy_dbg("[IMG_VID] Syncing latest ComfyUI & GGUF nodes...")
    subprocess.run(
        "git clone --depth 1 -q https://github.com/Comfy-Org/ComfyUI.git /tmp/c_latest && "
        "cp -r /tmp/c_latest/* /kaggle/working/ComfyUI/ 2>/dev/null || true; "
        "mkdir -p /kaggle/working/ComfyUI/custom_nodes/ComfyUI-GGUF && "
        "git clone --depth 1 -q https://github.com/leejet/ComfyUI-GGUF.git /tmp/g_latest && "
        "cp -r /tmp/g_latest/* /kaggle/working/ComfyUI/custom_nodes/ComfyUI-GGUF/ 2>/dev/null || true; "
        "rm -rf /tmp/c_latest /tmp/g_latest",
        shell=True,
        check=False,
    )

    print("📦 [1/2] Cài đặt offline cloudflared (.deb)...")
    subprocess.run(f"dpkg -i --force-depends {rt_dir}/debs/*.deb >/dev/null 2>&1", shell=True, check=False)
    if not (shutil.which("cloudflared") or os.path.exists("/usr/local/bin/cloudflared")):
        subprocess.run("curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /tmp/cloudflared && chmod +x /tmp/cloudflared", shell=True, check=False)

    print("📦 [2/2] Cài đặt offline toàn bộ ComfyUI Python wheels (--no-index --no-deps)...")
    _ntfy_dbg("[IMG_VID] Installing offline wheels...")
    res_pip = subprocess.run(
        f"{sys.executable} -m pip install -q --no-index --no-deps --find-links={rt_dir}/wheels {rt_dir}/wheels/*.whl",
        shell=True,
        check=False,
    )
    if res_pip.returncode != 0:
        for whl in sorted(glob.glob(f"{rt_dir}/wheels/*.whl")):
            subprocess.run(f"{sys.executable} -m pip install -q --no-index --no-deps {whl}", shell=True, check=False)
    shutil.rmtree(rt_dir, ignore_errors=True)
else:
    print("⚠️ Không tìm thấy bundle, tải trực tiếp...")
    _ntfy_dbg("[IMG_VID] No bundle found, fallback setup...")
    subprocess.run(
        "git clone -q --depth 1 https://github.com/Comfy-Org/ComfyUI.git /kaggle/working/ComfyUI && "
        "mkdir -p /kaggle/working/ComfyUI/custom_nodes && "
        "git clone -q --depth 1 https://github.com/leejet/ComfyUI-GGUF.git /kaggle/working/ComfyUI/custom_nodes/ComfyUI-GGUF && "
        "git clone -q --depth 1 https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git /kaggle/working/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite",
        shell=True,
        check=False,
    )
    subprocess.run("curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /tmp/cloudflared && chmod +x /tmp/cloudflared", shell=True, check=False)

# Đảm bảo các gói FastAPI cốt lõi không xung đột
try:
    import fastapi, uvicorn, yaml, requests, psutil
except Exception as imp_err:
    print(f"📦 Bổ sung nhanh các gói phụ trợ FastAPI ({imp_err})...")
    _ntfy_dbg(f"[IMG_VID] Installing missing FastAPI deps ({imp_err})...")
    subprocess.run(
        f"{sys.executable} -m pip install -q --prefer-binary fastapi uvicorn pyyaml python-multipart requests pillow psutil comfy-aimdo comfy-kitchen torchsde einops spandrel kornia scipy",
        shell=True,
        check=False,
    )

with open("/tmp/.comfyui_deps_ready", "w") as mf:
    mf.write("ok")

msg = f"[IMG_VID] Cell 3 setup completed in {time.time() - t_setup_0:.1f}s!"
print("✅", msg)
_ntfy_dbg(msg)


In [ ]:
# 4. Khởi động Studio (image_video mode - Hỗ trợ Intentional Shutdown exit 99)
import os, subprocess, sys, time, urllib.request

def _ntfy(msg: str):
    for t in ["studio-ai-url-cuongnguyen1802", "studio-ai-url-nguynxuncngde180528", "studio-ai-debug-cuongnguyen1802"]:
        try:
            req = urllib.request.Request(f"https://ntfy.sh/{t}", data=msg.encode("utf-8"), headers={"Title": "Visual Studio Main", "Priority": "4"}, method="POST")
            urllib.request.urlopen(req, timeout=4)
        except Exception:
            pass

os.environ["STUDIO_MODE"] = "image_video"
_ntfy("🚀 [IMG_VID] Cell 4 starting main.py --mode image_video...")
for attempt in range(3):
    t0 = time.time()
    ret = subprocess.run([sys.executable, "-u", "main.py", "--mode", "image_video"], cwd=os.getcwd())
    if ret.returncode in (0, 99):
        print(f"Clean exit requested ({ret.returncode}). Stopping notebook.")
        _ntfy(f"🏁 Clean exit requested ({ret.returncode}). Server stopped.")
        break
    if time.time() - t0 < 15:
        print(f"Fast crash (code {ret.returncode}). Exiting to preserve error log.")
        _ntfy(f"❌ Fast crash detected (code {ret.returncode}) after {time.time() - t0:.1f}s.")
        break
    print(f"Server exited with code {ret.returncode}, restarting in 3s...")
    _ntfy(f"⚠️ Server exited (code {ret.returncode}), restarting...")
    time.sleep(3)
